# GF4 W4A4 + Hessian: adaptive & fixed clip (Mistral / Qwen)

Runs the **real GF4 pipeline** (`llm_quant_eval.py`: randomized-Hadamard rotation, NVFP4/E2M1 weights via **Hessian reconstruction**, GF4 activations) on Mistral-7B and Qwen2.5-7B/14B, to fill the W4A4 rows of the main table for these models. Two forms per model:

- **adaptive + Hessian** — PER-BLOCK online activation clip (`--per-block-clip`): each 32-elem block picks the clip minimizing its own reconstruction MSE, every forward, no calibration. Matches the original FP\_Quant experiments' `quantize_activations_gf4_adaptive` (bit\_split.py) — the 'alpha per block' iso-configuration.
- **fixed + Hessian** — a single pinned clip ratio (`--fixed-clip 2.5`); weight-side Hessian reconstruction unchanged

**Block size:** the weight Hessian/E2M1 reconstruction uses `--hess-block 16`, matching the deployed 'pure GF4' pipeline. Note the GF4 *activation* encoder in this CUDA harness is compiled at block 32 (a warp-32 reduction), so activations here are block-32 while weights are block-16; for fully block-16 activations use the deployed FP\_Quant pipeline. The weight-side Hessian — the part your correction targets — is block-16 here.

**Memory-aware:** `--device-map` spreads each model across GPU+CPU, and each model's HF cache is **purged before the next** so peak disk stays near the largest single model (~28 GB for 14B), not the sum. Result CSVs are written per-config so an OOM never loses finished rows.

Set **Runtime → A100 GPU, High-RAM**. (Qwen2.5-14B needs the 40 GB A100; on a smaller GPU add `--offload-folder /content/off` inside `eval_run`.)

In [ ]:
!pip install -q transformers datasets accelerate scipy sentencepiece ninja
# ninja is REQUIRED for torch to JIT-compile the CUDA kernels.
# Mistral / Qwen are open -- no HF login needed.

In [ ]:
# --- point Colab at your CUDA_FP4_Test folder -------------------------------
# Upload it (Files pane) OR clone your repo, so THESE files sit in the CWD:
#   bindings.cpp  hadamard_kernel.cu  gf4_encode_kernel.cu
#   e2m1_fused_gemv_kernel.cu  hessian_weight_quant_kernel.cu
#   gf4_common.cuh  e2m1_common.cuh  reference.py  llm_quant_eval.py
import os
WORKDIR = "CUDA_FP4_Test"          # <-- change if you uploaded elsewhere (e.g. "." )
if os.path.isdir(WORKDIR): os.chdir(WORKDIR)
need = ["bindings.cpp","hadamard_kernel.cu","gf4_encode_kernel.cu",
        "e2m1_fused_gemv_kernel.cu","gf4_fused_gemv_kernel.cu",
        "hessian_weight_quant_kernel.cu","gf4_common.cuh","e2m1_common.cuh",
        "reference.py","llm_quant_eval.py"]
missing = [f for f in need if not os.path.exists(f)]
print("cwd:", os.getcwd())
print("MISSING:", missing if missing else "none -- ready")
assert not missing, "upload/clone the CUDA_FP4_Test files into the working directory first"

In [ ]:
# ======================= CONFIG =======================
MODELS = [
    "mistralai/Mistral-7B-v0.1",
    "Qwen/Qwen2.5-7B",
    "Qwen/Qwen2.5-14B",
]
EVAL_WINDOWS = 10000      # FULL WikiText-2 test set: get_windows caps to total//seqlen
                          # (~120-150 windows). Matches the deployed OPT/LLaMA full-set
                          # protocol (compute_ppl_gptq_style) so columns are comparable to
                          # Table 1 and to the literature. Was 40 (an easier first-N subset:
                          # e.g. Qwen2.5-14B FP16 4.60 on 40 windows vs ~5.25 full-set).
FIXED_CLIP   = 2.5        # clip ratio for the fixed-clip + Hessian form
HESS_BLOCK   = 16         # weight Hessian/E2M1 block = deployed "pure GF4" block size
RUN_BASELINE = True       # FP16 ppl (set False if you already have it from the suite)
# ======================================================
import subprocess, os, shutil

# CSVs are APPEND-mode (llm_quant_eval.py:_append_result). Clear them at the start
# of a full run so stale rows from earlier (debug) runs can't poison the summary --
# the SHOW cell reads the LAST row per model, but a clean file removes all doubt.
for _csv in ("gf4_hessian_baseline.csv", "gf4_hessian_adaptive.csv", "gf4_hessian_fixed.csv"):
    if os.path.exists(_csv):
        os.remove(_csv); print("[cleared stale CSV]", _csv)

def disk_free_gb():
    try: return shutil.disk_usage("/").free / 1e9
    except Exception: return float("nan")

def purge_model(m):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE as base
    except Exception:
        base = os.path.expanduser("~/.cache/huggingface/hub")
    d = os.path.join(base, "models--" + m.replace("/", "--"))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True); print("  [purged cache]", d)

def eval_run(model, extra, out_csv, config="w4a4"):
    cmd = ["python", "llm_quant_eval.py", "--model", model, "--config", config,
           "--num-eval-windows", str(EVAL_WINDOWS), "--device-map",
           "--hess-block", str(HESS_BLOCK),
           "--results-csv", out_csv] + extra
    print("  $", " ".join(cmd), flush=True)
    # Stream the child's output line-by-line (subprocess.run's inherited stdout is
    # NOT shown reliably in Colab), and fail LOUDLY on a nonzero exit.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        print(f"  !! FAILED (exit {p.returncode}) -- fix the error above before continuing", flush=True)
        raise RuntimeError(f"{model} [{config}] failed (exit {p.returncode})")

for m in MODELS:
    print(f"\n===== {m}   (disk free {disk_free_gb():.0f} GB) =====", flush=True)
    if RUN_BASELINE:
        eval_run(m, [], "gf4_hessian_baseline.csv", config="baseline")   # FP16 PPL
    eval_run(m, ["--per-block-clip"], "gf4_hessian_adaptive.csv")        # PER-BLOCK adaptive clip + Hessian
    eval_run(m, ["--fixed-clip", str(FIXED_CLIP)], "gf4_hessian_fixed.csv")  # fixed clip + Hessian
    purge_model(m)                                                       # free disk before next model
    print(f"  [done {m}; disk free {disk_free_gb():.0f} GB]", flush=True)
print("\nALL DONE")

In [ ]:
# Collect the three result CSVs into one view.
import csv, os
def load(path):
    if not os.path.exists(path): return []
    with open(path) as f: return list(csv.DictReader(f))
rows = {"baseline": load("gf4_hessian_baseline.csv"),
        "adaptive+Hessian": load("gf4_hessian_adaptive.csv"),
        "fixed+Hessian": load("gf4_hessian_fixed.csv")}
print(f"{'model':30s} {'baseline':>10s} {'adaptive':>10s} {'fixed':>10s}")
models = sorted({r['model'] for rs in rows.values() for r in rs})
def ppl(rs, m):
    # LAST match: CSVs are append-mode, so the most recent run is the last row.
    val = float('nan')
    for r in rs:
        if r['model'] == m: val = float(r['ppl'])
    return val
for m in models:
    b = ppl(rows['baseline'], m); a = ppl(rows['adaptive+Hessian'], m); f = ppl(rows['fixed+Hessian'], m)
    print(f"{m:30s} {b:10.4f} {a:10.4f} {f:10.4f}   "
          f"dPPL(adaptive) {a-b:+.3f}   dPPL(fixed) {f-b:+.3f}")
print("\n(also saved: gf4_hessian_{baseline,adaptive,fixed}.csv)")